# Supervised test-time adaptation (teaching analogue)

**MedLIME Learning Track — Notebook 09/12** · Curriculum weeks 25–28 · Suggested time: 60–100 minutes

**Before starting:** Notebook 08; gradients and binary cross-entropy.

**Learning goal:** Optimize a lightweight input transform with a frozen model and examine the supervision assumptions.

> **Important:** All images below are synthetic visual teaching phantoms, **not real medical images**. The educational implementations are **not** the original MedLIME implementation, are not evaluated for clinical use, and do not establish diagnostic or explanation reliability.


In [ ]:
from pathlib import Path
import sys
here = Path.cwd()
for candidate in (here, here.parent, here.parent.parent):
    if (candidate / 'medlime_tutorials.py').exists():
        sys.path.insert(0, str(candidate.resolve())); break
else:
    raise FileNotFoundError('Open Jupyter from the MedLIME folder or its notebooks/ subfolder')
import numpy as np
import matplotlib.pyplot as plt
from medlime_tutorials import *
np.set_printoptions(precision=3, suppress=True)


## What adaptation means here
We **freeze** the predictor and optimize just scale and bias, using a labeled, darkened *synthetic* query. This uses the predictor's gradient during adaptation even though subsequent LIME explanation queries remain black-box. The label is explicitly provided by the experimenter. This is a **deliberate simplification** of MedLIME STTA, whose architectural and supervision details must be read from the paper before attempting reproduction.

Objective (toy): binary cross entropy \(\ell(g(T_\phi(x)),y) + \eta\lVert\phi-\phi_0\rVert^2\).

In [ ]:
q,mask=synthetic_image(seed=18,lesion=True)
target=q*.58
black_box=ToyBlackBox()
print('Original probability',float(black_box.predict_proba(q)[0]))
print('Shifted probability',float(black_box.predict_proba(target)[0]))
adapter,history=supervised_adapt(target,true_label=1,predictor=black_box,steps=55)
print('Adapted transformation:',adapter)
print('Adapted probability',float(black_box.predict_proba(adapter.apply(target))[0]))
plt.plot(history);plt.xlabel('Optimization step');plt.ylabel('BCE + regularizer');plt.title('Toy supervised adaptation');plt.show()

## Can we interpret transformed predictions?
The adapter is fitted on the single query's label. This is optimistic supervision and must never be counted as an independent blind-test localization result. In real experiments, document how each supervised adaptation target becomes available.

In [ ]:
baseline=lime_explain(target,black_box,n=130,seed=17,fill='zero')
adapted=lime_explain(target,black_box,n=130,seed=17,fill='zero',adapt=adapter)
show_images([target,adapter.apply(target),baseline['map'],adapted['map']],
 ['dark query','adapted query','LIME before','LIME after adaptation'],cmap='viridis')
plt.show()
print('AUPRC before/after:',round(auprc(mask,baseline['map']),3),round(auprc(mask,adapted['map']),3))

## Stress test on a negative case
Unconstrained supervised adaptation of a positive case can shift all pixels upward, making an unrelated normal image appear positive. Avoid inferring calibration or clinical robustness from one-case training.

In [ ]:
normal,_=synthetic_image(seed=19,lesion=False)
print('Normal score without adapter:',round(float(black_box.predict_proba(normal*.58)[0]),5))
print('Normal score with positive-query adapter:',round(float(black_box.predict_proba(adapter.apply(normal*.58))[0]),5))

## What you should now understand
- During toy adaptation, model weights stay frozen; only input-transform parameters change.
- A differentiable optimization experiment can clarify why an objective changes predictions.
- True-label test-time supervision is a major assumption; never conceal it.

## Try it yourself (do not just read the code)
1. Try weight_decay=0.2 versus 0.01 and examine changes to transformed intensity.
2. Optimize with an intentionally incorrect label and report the resulting failure.

## Exit ticket
**Question:** Is a transformed image that increases confidence necessarily more faithful to the original pathology?

Record your answers in a new Markdown cell before moving to the next notebook. Many questions have more than one reasonable answer; explain your evidence.